In [62]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle




In [63]:
data = pd.read_csv('bank_churn.csv')
data.head()

,CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,100001,650,France,Female,42,5,125000,2,1,1,55000,0
1,100002,720,Germany,Male,35,8,145000,1,1,1,85000,0
2,100003,580,Spain,Female,52,3,80000,1,0,0,45000,1
3,100004,690,France,Male,29,7,0,2,1,1,65000,0
4,100005,500,Germany,Female,60,2,170000,1,1,0,55000,1


In [64]:
## preprocessing the data
### Drop irrelevant or non-numeric columns
data = data.drop(['CustomerId'], axis=1)


In [65]:
## Encode categorical variables
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,650,France,0,42,5,125000,2,1,1,55000,0
1,720,Germany,1,35,8,145000,1,1,1,85000,0
2,580,Spain,0,52,3,80000,1,0,0,45000,1
3,690,France,1,29,7,0,2,1,1,65000,0
4,500,Germany,0,60,2,170000,1,1,0,55000,1
5,790,Spain,1,40,10,90000,2,1,1,120000,0
6,610,France,0,48,4,115000,1,1,0,48000,1
7,680,Germany,1,31,6,130000,2,1,1,70000,0
8,540,Spain,0,55,1,150000,1,0,0,35000,1
9,710,France,1,38,8,0,2,1,1,95000,0


In [66]:
## One hot encoding 'Geography'
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo = OneHotEncoder()
geo_encoder = onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoder


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 50 stored elements and shape (50, 3)>

In [67]:
onehot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [68]:
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,1.0,0.0
2,0.0,0.0,1.0
3,1.0,0.0,0.0
4,0.0,1.0,0.0
5,0.0,0.0,1.0
6,1.0,0.0,0.0
7,0.0,1.0,0.0
8,0.0,0.0,1.0
9,1.0,0.0,0.0


In [69]:
data = pd.concat([data.drop(['Geography'], axis=1), geo_encoded_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,650,0,42,5,125000,2,1,1,55000,0,1.0,0.0,0.0
1,720,1,35,8,145000,1,1,1,85000,0,0.0,1.0,0.0
2,580,0,52,3,80000,1,0,0,45000,1,0.0,0.0,1.0
3,690,1,29,7,0,2,1,1,65000,0,1.0,0.0,0.0
4,500,0,60,2,170000,1,1,0,55000,1,0.0,1.0,0.0


In [70]:
#save the encoders and scaler 

with open('label_encoder_gender.pkl', 'wb') as f:
    pickle.dump(label_encoder_gender, f)

with open('onehot_encoder_geo.pkl', 'wb') as f:
    pickle.dump(onehot_encoder_geo, f)  

In [72]:
## divide the dataset into independent and dependent features 

X = data.drop('Exited', axis=1)
y = data['Exited']

## Split the data in training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## scale these features 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
12,580,0,51,2,140000,1,0,0,42000,1.0,0.0,0.0
4,500,0,60,2,170000,1,1,0,55000,0.0,1.0,0.0
37,730,1,36,8,102000,2,1,1,89000,0.0,0.0,1.0
8,540,0,55,1,150000,1,0,0,35000,0.0,0.0,1.0
3,690,1,29,7,0,2,1,1,65000,1.0,0.0,0.0
6,610,0,48,4,115000,1,1,0,48000,1.0,0.0,0.0
41,780,1,33,10,87000,2,1,1,119000,1.0,0.0,0.0
46,570,0,55,2,162000,1,0,0,36000,0.0,0.0,1.0
47,720,1,34,8,97000,2,1,1,86000,1.0,0.0,0.0
15,810,1,41,10,85000,2,1,1,130000,1.0,0.0,0.0


In [73]:
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [79]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime 

In [80]:
X_train.shape[1]

12

In [83]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid') ##output layer
    

])

In [87]:
import tensorflow as tf
opt = tf.keras.optimizers.Adam(learning_rate=0.01)
loss = tf.keras.losses.BinaryCrossentropy()
loss

<LossFunctionWrapper(<function binary_crossentropy at 0x1657202c0>, kwargs={'from_logits': False, 'label_smoothing': 0.0, 'axis': -1})>

In [89]:
model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])

In [91]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir = 'logs/fir' + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)


In [94]:
## EarlyStopping
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights= True)


In [95]:
history=model.fit(
    X_train, y_train, validation_data=(X_test, y_test), epochs=100,
    callbacks= [tensorflow_callback, early_stopping_callback]
)

Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 233ms/step - accuracy: 0.5250 - loss: 14320.2607 - val_accuracy: 0.8000 - val_loss: 1005.6436
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.7500 - loss: 1341.6422 - val_accuracy: 0.9000 - val_loss: 300.1093
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.8500 - loss: 483.6293 - val_accuracy: 0.9000 - val_loss: 10.5932
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.9000 - loss: 167.2032 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.9250 - loss: 198.6874 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.9250 - loss: 274.4444 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.9500 - loss: 267.2908 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.9500 - loss: 1

In [ ]:
model.save('model.h5')